# Efficient LLM Serving at Scale with Unified Caching (vLLM + LMCache)

**The idea:** vLLM's prefix cache lives in GPU memory. When many long requests overflow that GPU pool, their KV gets **evicted and recomputed** on every repeat. LMCache adds a CPU-DRAM tier: evicted KV is **reloaded from host memory** instead of recomputed — cutting time-to-first-token (TTFT).

**We run the same overflow workload twice (cold then warm) under two servers:**

| Server | Warm-pass TTFT |
|---|---|
| vLLM prefix cache only | stays high — working set exceeds GPU pool, recomputed every time |
| vLLM prefix cache **+ LMCache** | **drops sharply** — evicted KV reloaded from CPU |

Workload: 24 long requests (ISL 32K) at concurrency 4 — large enough to overflow the (deliberately small) GPU KV pool. Model **`Qwen/Qwen3-8B`**, running **inside `vllm/vllm-openai-rocm:v0.23.0`** on one MI300X.

> **⚠️ Critical for AMD/ROCm:** `pip install lmcache` ships **CUDA-only** wheels — its compiled `c_ops` fails to load on ROCm (`libcudart.so.13: cannot open shared object file`) and LMCache silently falls back to a ~50× slower Python/torch copy path, so the warm pass ends up *slower* than recompute. You **must build LMCache from source with `BUILD_WITH_HIP=1`** (Cell 1). With the HIP build, the warm pass on Qwen3-8B drops from ~13 s to ~0.3 s (**~39× faster**).

## 1. Build LMCache for ROCm (`BUILD_WITH_HIP=1`) + the AMD fixes

Stock image has no LMCache, and **`pip install lmcache` is the wrong path on AMD** — its prebuilt `c_ops` is CUDA-only (`libcudart.so.13`) and silently disables the fast KV-transfer backend on ROCm, so LMCache falls back to a ~50× slower torch copy path and the warm pass ends up *slower* than recompute.

The fix (verified on MI300X, `vllm/vllm-openai-rocm:v0.23.0`):

1. **Clone LMCache and build from source with `BUILD_WITH_HIP=1`** — compiles the HIP `c_ops`. Needs the ROCm toolchain (`hipcc`), which the vLLM-ROCm image already ships. Takes ~3–5 min.
2. **Force-reinstall the ROCm CuPy** (`cupy-rocm-7-0`) — a plain install can leave CuPy half-removed (`No module named cupy_backends`); ROCm CuPy is also what stops the cache server hanging at startup.
3. **Re-pin `grpcio==1.78.0` and `numpy==2.1.3`** — the source build and CuPy pull in versions that break vLLM.

This is idempotent: if a HIP `c_ops` already imports, the build is skipped. The verify block must print **`c_ops loaded OK`** and **`is_hip = True`** — otherwise you are on the slow fallback path and the warm-pass win will not appear.

In [ ]:
%%bash
set -e
LMCACHE_VERSION=v0.5.0

# 1) Build LMCache from source WITH HIP (compiles the ROCm c_ops; pip wheels are CUDA-only).
#    Skip if a working HIP c_ops already imports (idempotent re-runs).
if python3 -c "from lmcache import c_ops" 2>/dev/null; then
  echo "HIP c_ops already present - skipping build"
else
  command -v hipcc >/dev/null || { echo "ERROR: hipcc not found - need the ROCm toolchain to build"; exit 1; }
  pip uninstall -y lmcache >/dev/null 2>&1 || true
  cd /tmp && rm -rf LMCache
  git clone --depth 1 --branch "$LMCACHE_VERSION" https://github.com/LMCache/LMCache.git 2>&1 | tail -1
  cd LMCache
  echo "building LMCache $LMCACHE_VERSION with BUILD_WITH_HIP=1 (~3-5 min)..."
  BUILD_WITH_HIP=1 pip install -e . --no-build-isolation 2>&1 | tail -2
fi

# 2) ROCm CuPy (force-reinstall so it isn't left half-removed)
pip uninstall -y cupy cupy-cuda12x cupy-cuda13x nixl nixl-cu12 nixl-cu13 nixl_ep >/dev/null 2>&1 || true
pip install --force-reinstall --no-cache-dir cupy-rocm-7-0 2>&1 | tail -1

# 3) Re-pin deps the source build / cupy pull in (grpcio must match vLLM; numpy 2.5 breaks the image)
pip install -q "numpy==2.1.3" "grpcio==1.78.0" 2>&1 | tail -1 || true

echo "---- verify (must show 'c_ops loaded OK' and is_hip = True) ----"
python3 -c "
import lmcache
from lmcache import c_ops              # ImportError here => still on the slow CUDA-wheel fallback
print('c_ops loaded OK')
import cupy
from cupy_backends.cuda.api import runtime as r
assert getattr(r, 'is_hip', False), 'CuPy is not the ROCm build (is_hip=False)'
print('lmcache', lmcache.__version__, '| cupy', cupy.__version__, '| is_hip =', r.is_hip)
"

## 1b. Fetch the model (one-time)

Downloads `Qwen/Qwen3-8B` to `/models/Qwen3-8B` if not already present. Qwen3 is ungated but the download still needs a valid `HF_TOKEN`. Skip this cell if the model is already staged.

In [ ]:
%%bash
if [ ! -f /models/Qwen3-8B/config.json ]; then
  export HF_TOKEN="${HF_TOKEN:?set HF_TOKEN in the environment before running}"
  hf download Qwen/Qwen3-8B --local-dir /models/Qwen3-8B 2>&1 | tail -2
else
  echo "Qwen3-8B already present at /models/Qwen3-8B"
fi

## 2. Baseline server — vLLM prefix cache only

`--gpu-memory-utilization 0.5` keeps the GPU KV pool small (~574K tokens for Qwen3-8B) so the 24×32K workload (~768K tokens) overflows it. `PYTHONHASHSEED=0` is required for cache-key consistency. Launches in the background; the next cell waits for it.

In [ ]:
%%bash
pkill -f "vllm serve" 2>/dev/null && sleep 4 || true
pkill -f "lmcache server" 2>/dev/null || true

export PYTHONHASHSEED=0
nohup vllm serve /models/Qwen3-8B \
  --served-model-name Qwen3-8B \
  --tensor-parallel-size 1 \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.5 \
  --enable-prefix-caching \
  > /tmp/vllm.log 2>&1 &
echo "launching prefix-only server (log: /tmp/vllm.log); first boot ~2-3 min" 

## 3. Wait for the server (live log)

Polls `/v1/models` and prints the tail of the server log each attempt, so you can watch weight-load → compile → ready.

In [ ]:
import time, requests, subprocess
for i in range(80):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print("\n==> server ready"); break
    except requests.RequestException:
        pass
    tail = subprocess.run(["tail","-n","3","/tmp/vllm.log"], capture_output=True, text=True).stdout
    print(f"--- waiting {i+1}/80 ---\n{tail}", flush=True)
    time.sleep(10)
else:
    raise RuntimeError("server did not start - see /tmp/vllm.log")

## 4. Benchmark the baseline (two passes)

`vllm bench serve` with a fixed `--seed`, run twice: **PASS 1 (cold)** populates, **PASS 2 (warm)** re-sends the identical prompts. With prefix-cache only and a working set (24×32K ≈ 768K tokens) that overflows the GPU pool (~574K), **PASS 2 ≈ PASS 1** — no benefit, because evicted KV is recomputed.

In [ ]:
import subprocess, re

def bench():
    cmd = ["vllm","bench","serve",
        "--model","Qwen3-8B","--tokenizer","/models/Qwen3-8B",
        "--base-url","http://localhost:8000","--endpoint","/v1/completions",
        "--dataset-name","random","--random-input-len","32000","--random-output-len","64",
        "--num-prompts","24","--max-concurrency","4","--seed","555","--ignore-eos",
        "--percentile-metrics","ttft"]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    m = re.search(r"Mean TTFT \(ms\):\s*([0-9.]+)", out)
    return float(m.group(1))/1000 if m else None

prefix_cold = bench(); print(f"prefix-only  PASS1 cold: {prefix_cold:6.2f} s")
prefix_warm = bench(); print(f"prefix-only  PASS2 warm: {prefix_warm:6.2f} s   ({prefix_cold/prefix_warm:.2f}x)")

## 5. Restart with LMCache — prefix cache + CPU KV offload

A separate `lmcache server` holds KV in CPU DRAM; vLLM talks to it via `LMCacheMPConnector`. **Both** processes get `PYTHONHASHSEED=0`. Same model, same GPU budget — only the CPU cache tier is added.

In [ ]:
%%bash
pkill -f "vllm serve" 2>/dev/null && sleep 4 || true
pkill -f "lmcache server" 2>/dev/null && sleep 2 || true

# CPU-DRAM KV cache server (l1-size-gb must fit host RAM; 150 GB is safe on a 235 GB box)
export PYTHONHASHSEED=0
nohup lmcache server \
  --host 127.0.0.1 --port 5555 \
  --l1-size-gb 150 --eviction-policy LRU --chunk-size 256 \
  > /tmp/lmcache_server.log 2>&1 &
sleep 6

# vLLM pointed at the LMCache server
export PYTHONHASHSEED=0
nohup vllm serve /models/Qwen3-8B \
  --served-model-name Qwen3-8B \
  --tensor-parallel-size 1 \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.5 \
  --enable-prefix-caching \
  --kv-transfer-config '{"kv_connector":"LMCacheMPConnector","kv_role":"kv_both"}' \
  > /tmp/vllm.log 2>&1 &
echo "launching prefix+LMCache server; first boot ~2-3 min" 

## 6. Wait for the LMCache server

In [ ]:
for i in range(80):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=5).ok:
            print("\n==> server ready"); break
    except requests.RequestException:
        pass
    tail = subprocess.run(["tail","-n","3","/tmp/vllm.log"], capture_output=True, text=True).stdout
    print(f"--- waiting {i+1}/80 ---\n{tail}", flush=True)
    time.sleep(10)
else:
    raise RuntimeError("server did not start - see /tmp/vllm.log")

## 7. Benchmark with LMCache (same two passes)

Identical workload. Now **PASS 2 ≪ PASS 1** — evicted KV is reloaded from CPU instead of recomputed. On Qwen3-8B with the HIP-built LMCache, the warm pass drops from ~13 s to well under 1 s (**tens of × faster**).

In [ ]:
lmcache_cold = bench(); print(f"LMCache  PASS1 cold: {lmcache_cold:6.2f} s")
lmcache_warm = bench(); print(f"LMCache  PASS2 warm: {lmcache_warm:6.2f} s   ({lmcache_cold/lmcache_warm:.2f}x)")

## 8. Proof it was the cache

LMCache 0.5.0 exposes cache activity on its Prometheus endpoint (`:8080/metrics`). On the warm pass, `lmcache_mp_lookup_hit_tokens_total` climbs above 0 — evicted KV was **reloaded from CPU**, not recomputed. If `requested` grows but `hit` stays 0 → cache keys mismatch (check `PYTHONHASHSEED=0` on both processes).

In [ ]:
%%bash
# lmcache 0.5.0 reports cache activity via its Prometheus endpoint (:8080/metrics),
# not the server log. lookup_hit_tokens going > 0 on the warm pass is the proof the
# evicted KV was reloaded from CPU instead of recomputed.
echo "---- store activity (server log) ----"
grep -cE "Stored [0-9]+ tokens" /tmp/lmcache_server.log | sed 's/^/store ops: /'
echo "---- lookup hits vs requests (metrics) ----"
curl -s http://localhost:8080/metrics \
  | grep -E "lmcache_mp_lookup_(hit|requested)_tokens_total" \
  | grep -v '^#' || echo "metrics endpoint not reachable"

## 9. Side-by-side result

Cold bars are similar (both recompute on first pass). The **warm bars are the story**: prefix-only stays high (recomputes evicted KV), prefix+LMCache drops (reloads from CPU).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])

import matplotlib.pyplot as plt
import numpy as np

groups = ["Cold (PASS 1)", "Warm (PASS 2)"]
x = np.arange(2); w = 0.35
fig, ax = plt.subplots(figsize=(6.5, 4))
b1 = ax.bar(x - w/2, [prefix_cold, prefix_warm], w, label="prefix cache only", color="#c0504d")
b2 = ax.bar(x + w/2, [lmcache_cold, lmcache_warm], w, label="prefix + LMCache", color="#4f81bd")
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylabel("Mean TTFT (s)")
ax.set_title("Warm pass: LMCache reloads evicted KV instead of recomputing")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{b.get_height():.1f}s",
                ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
print(f"Warm TTFT: prefix-only {prefix_warm:.1f}s vs LMCache {lmcache_warm:.1f}s "
      f"-> {prefix_warm/lmcache_warm:.1f}x faster")

## 10. When does LMCache help?

Only when **both** hold:

| Condition | Why |
|---|---|
| **Long context reused** | Reload only pays off if recompute (long prefill) is expensive. |
| **GPU HBM under pressure** | If the working set fits in GPU, vLLM's prefix cache already serves reuse and LMCache just adds transfer cost. |

That's why this demo overflows a small GPU pool with long requests. With a single short request, or a large GPU pool, prefix-cache alone is enough and LMCache shows no benefit — that's expected.

### AMD / ROCm gotchas (each is a separate, silent failure mode)
- **Build LMCache with `BUILD_WITH_HIP=1`** (Cell 1) — the #1 issue. `pip install lmcache` ships a CUDA-only `c_ops` that fails to import on ROCm; LMCache then falls back to a ~50× slower torch copy path and the warm pass is *slower* than recompute. Verify `c_ops loaded OK`.
- **`cupy-rocm-7-0` (force-reinstall)** — required, or the cache server hangs at startup. `is_hip` must be `True`.
- **`PYTHONHASHSEED=0` on every process** — required, or 0 % cache hits (silent). Watch the metric below.
- **`LMCacheMPConnector`** (separate server) — use on ROCm; the in-engine connector faults under concurrency.
- **`--l1-size-gb` must fit host RAM** — it pins that many GiB up front. On a 235 GB box, `1000` aborts the server with `DefaultCPUAllocator: can't allocate`; keep it ≤ ~half of RAM.

### Sanity check: hardware bandwidth is *not* the bottleneck
Raw GPU↔CPU DMA on this MI300X (measured with a pinned-memory `torch` copy) is **~52 GB/s** — identical on a virtualized VF and on bare metal. If LMCache seems slow (effective offload ≈ 1 GB/s), that is the **CUDA-wheel fallback path**, not the bus. The fix is the HIP build, not more bandwidth.

### Model note: use a standard-attention model
This demo uses **Qwen3-8B**. Models with **sliding-window attention + heterogeneous head_dim** (e.g. gemma-4) **store** KV into LMCache but return **0 lookup hits** (cache-key/layout mismatch), so they show no warm-pass win even with the correct HIP build. Qwen3, Llama3, and Qwen-VL work well (see the AMD × LMCache blog).